In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
import warnings
warnings.filterwarnings('ignore')

# Загрузка датасета

Датасет представляет собой несколько xlsx файлов. Каждый год - каждый отдельный файл. 

Для тестов был взят 2025 год. 

В каждом файле находятся различные листы данных, которые загружаются в виде единого dataframe c dataframe в виде колонок. Для удобства они будут вынесены в отдельные переменные

In [2]:
data = pd.read_excel("./data_raw/data.xlsx", sheet_name=None)

In [3]:
df_first = data["Больные"]
df_second = data["Монитор_1"]
df_third = data["Монитор_2"]
df_forth = data["Монитор_3"]
df_fifth = data["ИВЛ"]
df_sixth = data["Балансы"]
df_seventh = data["Лаб"]
df_eights = data["Препараты"]
df_nines = data["ЗППТ"]

# Дата

Каждый датафрейм имеет колонку, в которой хранится дата получения наблюдения. Важно преобразовывать ее в дату для корректной обработки. 

In [ ]:
df_second["Время поступления"] = pd.to_datetime(df_second["Время поступления"])
df_second["chartTime"] = pd.to_datetime(df_second["chartTime"])
df_third["Время поступления"] = pd.to_datetime(df_third["Время поступления"])
df_third["chartTime"] = pd.to_datetime(df_third["chartTime"])
df_forth["Время поступления"] = pd.to_datetime(df_forth["Время поступления"])
df_forth["chartTime"] = pd.to_datetime(df_forth["chartTime"])
df_fifth["Время поступления"] = pd.to_datetime(df_fifth["Время поступления"])
df_fifth["chartTime"] = pd.to_datetime(df_fifth["chartTime"])
df_sixth["Время поступления"] = pd.to_datetime(df_fifth["Время поступления"]) # Данные снимаются раз в сутки
df_seventh["Время поступления"] = pd.to_datetime(df_seventh["Время поступления"])
df_seventh["chartTime"] = pd.to_datetime(df_seventh["chartTime"])
df_eights["Время поступления"] = pd.to_datetime(df_eights["Время поступления"])
df_eights["chartTime"] = pd.to_datetime(df_eights["chartTime"])
df_nines["Время поступления"] = pd.to_datetime(df_nines["Время поступления"])
df_nines["Start"] = pd.to_datetime(df_nines["Start"])
df_nines["Stop"] = pd.to_datetime(df_nines["Stop"])


Ставим индекс по времени поступления

In [ ]:
df_second.set_index("Время поступления", inplace=True)
df_third.set_index("Время поступления", inplace=True)
df_forth.set_index("Время поступления", inplace=True)
df_fifth.set_index("Время поступления", inplace=True)
df_sixth.set_index("Время поступления", inplace=True)
df_seventh.set_index("Время поступления", inplace=True)
df_eights.set_index("Время поступления", inplace=True)
df_nines.set_index("Время поступления", inplace=True)

KeyError: "None of ['Время поступления'] are in the columns"

In [17]:
df_second.sort_index(inplace=True)
df_third.sort_index(inplace=True)
df_forth.sort_index(inplace=True)
df_fifth.sort_index(inplace=True)
df_sixth.sort_index(inplace=True)
df_seventh.sort_index(inplace=True)
df_eights.sort_index(inplace=True)
df_nines.sort_index(inplace=True)

In [18]:
df_second

,ID визита,Пациент,N иб,День пребывания,chartTime,"tº, C",АД,нАД,ЧСС,Пульс,...,ВГОК,ВСВЛ,МОК (УЗИ),"tº крови, C","Пульсация лучевой артерии, лев.","Пульсация лучевой артерии, прав.",Проверка CSM,Наполнение капилляров,Парадоксальный пульс,Лодыжечно-плечевой индекс
Время поступления,,,,,,,,,,,,,,,,,,,,,
2025-01-01 05:49:14.650,17624,Руденок,1,1.0,2025-01-01 06:00:00,36.2,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-01-01 05:49:14.650,17624,Руденок,1,2.0,2025-01-02 03:30:00,NaN,NaN,NaN,47.0,47.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-01-01 05:49:14.650,17624,Руденок,1,2.0,2025-01-02 03:00:00,NaN,NaN,NaN,52.0,52.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-01-01 05:49:14.650,17624,Руденок,1,2.0,2025-01-02 02:30:00,NaN,NaN,NaN,52.0,52.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-01-01 05:49:14.650,17624,Руденок,1,2.0,2025-01-02 02:20:00,NaN,NaN,43.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-08-29 14:20:46.670,19555,Зяблитсов,2599,1.0,2025-08-29 21:35:00,NaN,NaN,72.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-08-29 14:20:46.670,19555,Зяблитсов,2599,1.0,2025-08-29 21:45:00,NaN,NaN,74.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-08-29 14:20:46.670,19555,Зяблитсов,2599,1.0,2025-08-29 21:55:00,NaN,NaN,100.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Частота

Важно определить частоту и периодичность сбора данных.
Узнать автоматически можно через y.index.freq
Задать руками можно через y = y.asfreq('D')

Пропуски в индексе — пропущенные даты. Это критично для многих моделей.